In [15]:
import numpy as np

# ── DATA ──────────────────────────────────────────────────────────────────────
np.random.seed(42)
N = 2000

credit_score   = np.random.normal(650, 80, N)
debt_ratio     = np.random.uniform(0.1, 0.9, N)
income         = np.random.normal(60000, 20000, N)
missed_payments = np.random.poisson(1, N).astype(float)



log_odds = (
    -4.0
    + 0.015 * (700 - credit_score)
    + 3.0   * debt_ratio
    + 0.8   * missed_payments
    - 0.00002 * income
)
prob_default = 1 / (1 + np.exp(-log_odds))
y = (np.random.rand(N) < prob_default).astype(float)

X_raw = np.column_stack([credit_score, debt_ratio, income, missed_payments])
mu, sigma = X_raw.mean(0), X_raw.std(0)
X = (X_raw - mu) / sigma

split = int(0.8 * N)
X_train, X_val = X[:split], X[split:]
y_train, y_val = y[:split], y[split:]

print(f"Train: {X_train.shape}, Val: {X_val.shape}")
print(f"Default rate  train={y_train.mean():.2%}  val={y_val.mean():.2%}")


print (type(X))

Train: (1600, 4), Val: (400, 4)
Default rate  train=17.31%  val=15.75%
<class 'numpy.ndarray'>


In [16]:
# ── NETWORK DEFINITION ────────────────────────────────────────────────────────

def init_weights(layer_sizes):
    """
    layer_sizes = [4, 8, 1]
    means: 4 inputs → 8 hidden neurons → 1 output
    """
    weights = []
    for i in range(len(layer_sizes) - 1):
        fan_in  = layer_sizes[i]
        fan_out = layer_sizes[i + 1]
        
        # He initialisation — small random weights
        W = np.random.randn(fan_in, fan_out) * np.sqrt(2.0 / fan_in)
        b = np.zeros((1, fan_out))
        
        weights.append((W, b))
    return weights

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def relu(z):
    return np.maximum(0, z)

# Initialise our network: 4 inputs → 8 hidden → 1 output
layer_sizes = [4, 8, 1]
weights = init_weights(layer_sizes)

W1, b1 = weights[0]
W2, b2 = weights[1]

print(f"W1 shape: {W1.shape}")  # (4, 8)
print(f"b1 shape: {b1.shape}")  # (1, 8)
print(f"W2 shape: {W2.shape}")  # (8, 1)
print(f"b2 shape: {b2.shape}")  # (1, 1)
print(f"\nW1 (first row): {W1[0].round(3)}")

W1 shape: (4, 8)
b1 shape: (1, 8)
W2 shape: (8, 1)
b2 shape: (1, 1)

W1 (first row): [ 0.043  0.25   0.382  1.369 -0.386 -0.368 -0.602 -0.757]


In [17]:
# ── FORWARD PASS ──────────────────────────────────────────────────────────────

def forward(X, weights):
    """
    X shape: (batch_size, 4)
    Returns prediction and cache (we need cache for backprop later)
    """
    W1, b1 = weights[0]
    W2, b2 = weights[1]

    # Layer 1: Input → Hidden
    Z1 = X @ W1 + b1          # (N, 4) @ (4, 8) + (1, 8) = (N, 8)
    A1 = relu(Z1)              # apply activation → (N, 8)

    # Layer 2: Hidden → Output
    Z2 = A1 @ W2 + b2         # (N, 8) @ (8, 1) + (1, 1) = (N, 1)
    A2 = sigmoid(Z2)           # apply activation → (N, 1)

    cache = (X, Z1, A1, Z2, A2)
    return A2, cache

# Run forward pass on training data
A2, cache = forward(X_train, weights)

print(f"Prediction shape : {A2.shape}")
print(f"First 5 predictions : {A2[:5].flatten().round(3)}")
print(f"Actual first 5      : {y_train[:5]}")
print(f"Mean prediction     : {A2.mean():.3f}")

Prediction shape : (1600, 1)
First 5 predictions : [0.481 0.408 0.497 0.27  0.862]
Actual first 5      : [0. 0. 0. 0. 0.]
Mean prediction     : 0.664


In [18]:
# ── LOSS FUNCTION ─────────────────────────────────────────────────────────────

def binary_cross_entropy(y_true, y_pred):
    """
    y_true : actual labels        shape (N,)
    y_pred : network predictions  shape (N, 1)
    """
    # reshape y_true to match y_pred shape
    y_true = y_true.reshape(-1, 1)
    
    # clip predictions to avoid log(0) → infinity
    y_pred = np.clip(y_pred, 1e-7, 1 - 1e-7)
    
    # binary cross entropy formula
    loss = -np.mean(
        y_true * np.log(y_pred) +
        (1 - y_true) * np.log(1 - y_pred)
    )
    return loss

# Calculate loss on our random predictions
loss = binary_cross_entropy(y_train, A2)
print(f"Loss (random weights) : {loss:.4f}")

# What does perfect prediction look like?
perfect = binary_cross_entropy(y_train, y_train.reshape(-1,1))
print(f"Loss (perfect)        : {perfect:.4f}")

# What does worst prediction look like?
worst = binary_cross_entropy(y_train, 1 - y_train.reshape(-1,1))
print(f"Loss (worst)          : {worst:.4f}")

Loss (random weights) : 1.0750
Loss (perfect)        : 0.0000
Loss (worst)          : 16.1181


In [19]:
# Manual calculation for 3 people
people = [
    {"y_true": 1, "y_pred": 0.99, "label": "confident correct"},
    {"y_true": 1, "y_pred": 0.50, "label": "unsure"},
    {"y_true": 1, "y_pred": 0.01, "label": "confident wrong"},
]

for p in people:
    loss = -(p["y_true"] * np.log(p["y_pred"]) + 
             (1-p["y_true"]) * np.log(1-p["y_pred"]))
    print(f"{p['label']:20s} → loss = {loss:.2f}")

confident correct    → loss = 0.01
unsure               → loss = 0.69
confident wrong      → loss = 4.61


In [20]:
# ── BACKPROPAGATION ───────────────────────────────────────────────────────────

def backward(y_true, weights, cache):
    """
    y_true : actual labels  shape (N,)
    cache  : everything saved during forward pass
    """
    X, Z1, A1, Z2, A2 = cache
    W1, b1 = weights[0]
    W2, b2 = weights[1]
    N = X.shape[0]

    y_true = y_true.reshape(-1, 1)

    # ── LAYER 2 GRADIENTS (output layer) ──────────────────────────────────────
    dA2 = A2 - y_true                        # how wrong is output?      (N, 1)
    dW2 = (A1.T @ dA2) / N                   # how much did W2 cause it? (8, 1)
    db2 = np.mean(dA2, axis=0, keepdims=True) # how much did b2 cause it? (1, 1)

    # ── LAYER 1 GRADIENTS (hidden layer) ──────────────────────────────────────
    dA1 = dA2 @ W2.T                         # pass error back            (N, 8)
    dZ1 = dA1 * (Z1 > 0)                    # ReLU gradient              (N, 8)
    dW1 = (X.T @ dZ1) / N                   # how much did W1 cause it?  (4, 8)
    db1 = np.mean(dZ1, axis=0, keepdims=True) # how much did b1 cause it? (1, 8)

    grads = (dW1, db1, dW2, db2)
    return grads

# Run backprop
grads = backward(y_train, weights, cache)
dW1, db1, dW2, db2 = grads

print(f"dW1 shape : {dW1.shape}")
print(f"dW2 shape : {dW2.shape}")
print(f"\ndW2 (how much each hidden neuron contributed to error):")
print(dW2.flatten().round(4))
print(f"\ndW1 (how much each input contributed to error):")
print(dW1.round(4))

dW1 shape : (4, 8)
dW2 shape : (8, 1)

dW2 (how much each hidden neuron contributed to error):
[0.1501 0.2254 0.1812 0.5381 0.5366 0.1427 0.1261 0.272 ]

dW1 (how much each input contributed to error):
[[-1.100e-03 -1.000e-04 -4.400e-03  1.660e-02  2.370e-02 -4.620e-02
  -4.190e-02 -4.100e-03]
 [ 5.900e-02  5.310e-02 -5.400e-03  2.110e-02  1.595e-01 -9.360e-02
  -2.060e-02  1.060e-02]
 [-3.200e-02 -5.300e-03  2.300e-03  7.400e-03 -7.020e-02  6.310e-02
   2.920e-02  1.000e-04]
 [-2.410e-02  7.000e-03  5.200e-03 -6.600e-03 -1.403e-01  2.630e-02
   2.630e-02 -8.800e-03]]
